# Probability and Statistics with Python
## Using Google Colab

Welcome! In this tutorial you will explore fundamental probability concepts,
discrete and continuous random variables, and basic descriptive statistics
— all using Python in Google Colab.

**Prerequisites:** Notebook 1 (Introduction to Data Science with Python).

**Duration:** approximately 1–2 hours.

First, let's import all the libraries we will need.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import randint, bernoulli, binom, poisson, norm

np.random.seed(42)  # reproducible results
print('Libraries imported successfully!')

---
## 1. Basic Probability Concepts

**Probability** measures how likely an event is to occur. It is always
a number between 0 (impossible) and 1 (certain):

$$P(A) = \frac{\text{number of favourable outcomes}}{\text{total outcomes}}$$

Key terms:
- **Sample space (S)** — the set of all possible outcomes
- **Event (A)** — a subset of the sample space we care about
- **Complement (Aᶜ)** — everything *not* in A; P(Aᶜ) = 1 − P(A)

### 1.1 Example: Rolling a Fair Die

A fair die has six equally likely outcomes: {1, 2, 3, 4, 5, 6}.
Let's compute and verify basic probabilities by simulating 10 000 rolls.

In [ ]:
# Theoretical probabilities for a fair die
sample_space = [1, 2, 3, 4, 5, 6]
p_any_face    = 1 / len(sample_space)   # P(X = k) for any k
p_even        = 3 / 6                   # P(even) = P({2,4,6})
p_greater_4   = 2 / 6                   # P(X > 4) = P({5,6})

print(f'P(any face)  = {p_any_face:.4f}')
print(f'P(even)      = {p_even:.4f}')
print(f'P(X > 4)     = {p_greater_4:.4f}')

We can **verify** these theoretical probabilities by simulating many rolls
and counting how often each event occurs. As the number of trials grows,
the simulated proportion should get closer and closer to the true probability
— this is the **Law of Large Numbers**.

In [ ]:
# Simulate 10 000 die rolls and compare with theory
rolls = np.random.randint(1, 7, size=10_000)

sim_even      = np.mean(rolls % 2 == 0)
sim_greater_4 = np.mean(rolls > 4)

print('Simulated P(even)   :', sim_even,      ' | Theory:', round(p_even, 4))
print('Simulated P(X > 4)  :', sim_greater_4, ' | Theory:', round(p_greater_4, 4))

---
## 2. Discrete Random Variables

A **random variable** assigns a number to each outcome of a random process.
A **discrete** random variable takes countable, distinct values
(e.g., 0, 1, 2, …).

The **Probability Mass Function (PMF)** gives the probability of each value:
$P(X = x)$ for every possible $x$.

### 2.1 Bernoulli Distribution

The **Bernoulli** distribution models a single trial with two outcomes:
success (1) with probability $p$, and failure (0) with probability $1-p$.
Example: a single coin flip, or whether one email is opened.

In [ ]:
# Bernoulli(p=0.7): a student passes an exam with probability 0.7
p = 0.7
rv = bernoulli(p)

print('Bernoulli PMF:')
for x in [0, 1]:
    print(f'  P(X={x}) = {rv.pmf(x):.3f}')
print(f'Expected value E[X] = {rv.mean():.3f}')

# Bar chart of the PMF
plt.figure(figsize=(4, 3))
plt.bar([0, 1], [rv.pmf(0), rv.pmf(1)], color='steelblue', edgecolor='black')
plt.xticks([0, 1], ['Fail (0)', 'Pass (1)'])
plt.title('Bernoulli PMF (p=0.7)')
plt.ylabel('Probability')
plt.tight_layout()
plt.show()

### 2.2 Binomial Distribution

The **Binomial** distribution counts the number of successes in $n$
independent Bernoulli trials. For example: how many of 10 emails are opened
if each has a 20% open rate?

In [ ]:
# Binomial(n=10, p=0.2): number of emails opened out of 10
n, p = 10, 0.2
rv = binom(n, p)

print('Binomial PMF (first 6 values):')
for x in range(6):
    print(f'  P(X={x}) = {rv.pmf(x):.4f}')
print(f'\nE[X] = {rv.mean():.2f}  (expected opens)')

# PMF bar chart
xs = np.arange(0, n + 1)
plt.figure(figsize=(7, 4))
plt.bar(xs, rv.pmf(xs), color='mediumpurple', edgecolor='black')
plt.title(f'Binomial PMF (n={n}, p={p})')
plt.xlabel('Number of successes (X)')
plt.ylabel('P(X = x)')
plt.xticks(xs)
plt.tight_layout()
plt.show()

### 2.3 Poisson Distribution

The **Poisson** distribution counts the number of events in a fixed interval
when events occur independently at a constant average rate $\lambda$.
Example: the number of customers arriving at a café in 10 minutes.

In [ ]:
# Poisson(lambda=3): avg 3 customers per 10-minute window
lam = 3
rv  = poisson(mu=lam)

print(f'P(X=5)   = {rv.pmf(5):.4f}  (exactly 5 customers)')
print(f'P(X<=2)  = {rv.cdf(2):.4f}  (at most 2 customers)')
print(f'E[X]     = {rv.mean():.2f}')

# PMF bar chart
xs = np.arange(0, 12)
plt.figure(figsize=(7, 4))
plt.bar(xs, rv.pmf(xs), color='salmon', edgecolor='black')
plt.title(f'Poisson PMF (λ={lam})')
plt.xlabel('Number of events (X)')
plt.ylabel('P(X = x)')
plt.xticks(xs)
plt.tight_layout()
plt.show()

---
## 3. Continuous Random Variables

A **continuous** random variable can take any value in a range (e.g., height,
temperature). Because there are infinitely many possible values, we describe
it using a **Probability Density Function (PDF)**:

$$P(a \le X \le b) = \int_a^b f(x)\,dx$$

The probability is the **area under the curve** between $a$ and $b$.
The **CDF** $F(x) = P(X \le x)$ accumulates this area from the left.

### 3.1 Normal Distribution

The **Normal (Gaussian)** distribution is the classic bell-shaped curve,
controlled by its mean $\mu$ (center) and standard deviation $\sigma$ (spread).
Many real-world measurements — heights, exam scores, measurement errors —
follow a normal distribution.

In [ ]:
# Normal distribution: student exam scores  μ=70, σ=10
mu, sigma = 70, 10
rv = norm(loc=mu, scale=sigma)

# Probability of scoring between 60 and 80
p_60_80 = rv.cdf(80) - rv.cdf(60)
print(f'P(60 ≤ X ≤ 80) = {p_60_80:.4f}  ({p_60_80*100:.1f}%)')
print(f'P(X > 90)       = {rv.sf(90):.4f}  ({rv.sf(90)*100:.1f}%)')

Let's plot the **PDF** and shade the region corresponding to
$P(60 \le X \le 80)$ to make the area-under-the-curve idea concrete.

In [ ]:
# PDF plot with shaded probability region
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 300)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- PDF ---
axes[0].plot(x, rv.pdf(x), color='steelblue', linewidth=2)
x_fill = np.linspace(60, 80, 200)
axes[0].fill_between(x_fill, rv.pdf(x_fill), alpha=0.4, color='orange',
                     label=f'P(60≤X≤80) = {p_60_80:.2f}')
axes[0].set_title(f'Normal PDF  (μ={mu}, σ={sigma})')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('f(x)')
axes[0].legend()

# --- CDF ---
axes[1].plot(x, rv.cdf(x), color='coral', linewidth=2)
axes[1].axhline(0.5, color='gray', linestyle='--', label='F(x) = 0.5  (median)')
axes[1].set_title(f'Normal CDF  (μ={mu}, σ={sigma})')
axes[1].set_xlabel('Score')
axes[1].set_ylabel('F(x) = P(X ≤ x)')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 4. Basic Statistics

Statistics help us **summarize** a dataset with a few key numbers.

| Measure | What it tells you |
|---------|------------------|
| **Mean** | The arithmetic average — the "balance point" of the data |
| **Median** | The middle value when data is sorted — resistant to extreme values |
| **Mode** | The most frequent value or class |
| **Std Dev (σ)** | How spread out the values are around the mean |

### 4.1 Statistics on Raw Data

When we have individual data points we can compute these measures directly
with NumPy or Pandas.

In [ ]:
# Raw exam scores for a class of 20 students
exam_scores = np.array([55, 60, 62, 65, 68, 70, 70, 72, 75, 76,
                         78, 80, 82, 82, 85, 88, 90, 92, 95, 98])

print('=== Exam Scores ===')
print(f'Mean   : {np.mean(exam_scores):.2f}')
print(f'Median : {np.median(exam_scores):.2f}')
print(f'Std Dev: {np.std(exam_scores, ddof=1):.2f}')  # ddof=1 for sample std

# Visualize with a histogram
plt.figure(figsize=(7, 4))
plt.hist(exam_scores, bins=8, color='steelblue', edgecolor='black', alpha=0.8)
plt.axvline(np.mean(exam_scores),   color='red',    linestyle='--', label='Mean')
plt.axvline(np.median(exam_scores), color='orange', linestyle='--', label='Median')
plt.title('Distribution of Exam Scores')
plt.xlabel('Score')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()

### 4.2 Statistics from a Grouped Frequency Table

Often data is presented as a **grouped frequency table** — each row
covers an interval (class) and reports how many observations fall in it.
We use the **midpoint** of each class as a representative value.

Formulas:
$$\bar{x} = \frac{\sum f_i x_i}{\sum f_i}$$
$$\text{Median} = L + \frac{\frac{N}{2} - F}{f_m} \times c$$
$$\text{Mode} = L + \frac{f_m - f_1}{2f_m - f_1 - f_2} \times c$$

In [ ]:
# Grouped frequency table of student scores
data = {
    'Interval' : ['50-59', '60-69', '70-79', '80-89', '90-99'],
    'Frequency': [3, 5, 8, 6, 4],
}
df = pd.DataFrame(data)

# Class width and midpoints
df[['Lower', 'Upper']] = df['Interval'].str.split('-', expand=True).astype(int)
df['Midpoint'] = (df['Lower'] + df['Upper']) / 2
c = df['Lower'].iloc[1] - df['Lower'].iloc[0]  # class width
df

Now we apply the grouped-data formulas step by step.

In [ ]:
# --- Mean ---
df['f_x'] = df['Frequency'] * df['Midpoint']
N = df['Frequency'].sum()
mean_val = df['f_x'].sum() / N

# --- Median ---
df['Cumulative'] = df['Frequency'].cumsum()
median_idx  = df[df['Cumulative'] >= N / 2].index[0]
median_row  = df.loc[median_idx]
F_prev      = df.loc[median_idx - 1, 'Cumulative'] if median_idx > 0 else 0
median_val  = median_row['Lower'] + ((N/2 - F_prev) / median_row['Frequency']) * c

# --- Mode ---
modal_idx = df['Frequency'].idxmax()
modal_row = df.loc[modal_idx]
f1 = 0 if modal_idx == 0              else df.loc[modal_idx - 1, 'Frequency']
f2 = 0 if modal_idx == len(df) - 1   else df.loc[modal_idx + 1, 'Frequency']
fm = modal_row['Frequency']
mode_val = modal_row['Lower'] + ((fm - f1) / (2*fm - f1 - f2)) * c

print(f'Mean   = {mean_val:.2f}')
print(f'Median = {median_val:.2f}')
print(f'Mode   = {mode_val:.2f}')

Let's visualize the frequency distribution as a bar chart with the
three measures of central tendency marked.

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(df['Midpoint'], df['Frequency'], width=c*0.8,
        color='steelblue', edgecolor='black', alpha=0.8)
plt.axvline(mean_val,   color='red',    linestyle='--', label=f'Mean = {mean_val:.1f}')
plt.axvline(median_val, color='orange', linestyle='--', label=f'Median = {median_val:.1f}')
plt.axvline(mode_val,   color='green',  linestyle='--', label=f'Mode = {mode_val:.1f}')
plt.title('Grouped Frequency Distribution')
plt.xlabel('Score (Midpoint)')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()

---
## 5. Assignment

Complete **both** tasks below in new code cells.

---

### Task A — Grouped Frequency Table

You are given the following data on the number of hours students
studied per week:

| Interval | Frequency |
|----------|-----------|
| 0–4      | 5         |
| 5–9      | 10        |
| 10–14    | 12        |
| 15–19    | 8         |
| 20–24    | 5         |

1. Create a Pandas DataFrame for this table.
2. Compute the **mean**, **median**, and **mode** using the grouped-data formulas.
3. Plot a bar chart of the frequency distribution and mark the three measures.

---

### Task B — Random Variable Simulation

Choose **one** of the following distributions:
- Binomial(n=15, p=0.4)
- Poisson(λ=5)
- Normal(μ=50, σ=8)

1. Compute the **theoretical** mean and standard deviation using `scipy.stats`.
2. **Simulate** 1 000 samples from the distribution.
3. Compare the simulated mean and std to the theoretical values.
4. Plot the PMF or PDF.

> **Tip:** for the Normal distribution, use `rv.pdf(x)` for the PDF and
> `np.linspace(mu - 4*sigma, mu + 4*sigma, 300)` for the x-axis.

### Your Work Starts Here ⬇️

Add your code and markdown cells below. Good luck!

In [ ]:
# Task A — Step 1: Create the DataFrame



#### Task B — Your code here

In [ ]:
# Task B — Step 1: Choose a distribution and compute theoretical stats

